# Fine-Tuning for Maths — PoliMillionaire

Fine-tunes `Qwen3-4B-Thinking` on math reasoning data (QLoRA via Unsloth), then evaluates on the Maths competition.

## Before running

Upload these two items to your Google Drive inside a folder called `NLP/`:

```
MyDrive/
└── NLP/
    ├── millionaire_client/   ← copy the entire folder from the repo
    └── .env                  ← your credentials file (USER, PASSWORD, MAIL)
```

Then: **Runtime → Change runtime type → T4 GPU** (or A100 if available).

In [ ]:
# Mount Drive and set up paths — done first, this must be.
from google.colab import drive
drive.mount('/content/drive')

import sys, os

NLP_DIR    = '/content/drive/MyDrive/NLP'
CLIENT_DIR = f'{NLP_DIR}/millionaire_client'
ENV_FILE   = f'{NLP_DIR}/.env'
SAVE_PATH  = f'{NLP_DIR}/qwen3-math-finetuned'

assert os.path.isdir(CLIENT_DIR), f'millionaire_client not found at {CLIENT_DIR}'
assert os.path.isfile(ENV_FILE),  f'.env not found at {ENV_FILE}'

if NLP_DIR not in sys.path:
    sys.path.insert(0, NLP_DIR)

print('Drive mounted. Paths OK.')

In [ ]:
# Install base dependencies, we must.
import subprocess, sys
pkgs = ['python-dotenv', 'requests', 'datasets']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Base packages installed.')

In [ ]:
# Connect to the game server, we do.
import dotenv, os
dotenv.load_dotenv(ENV_FILE)

from millionaire_client import MillionaireClient, AuthenticationError

API_URL  = 'http://131.175.15.22:51111/'
username = os.getenv('USER')
pwd      = os.getenv('PASSWORD')

client = MillionaireClient(API_URL)
try:
    user = client.login(username, pwd)
    print(f'Login successful. Welcome, {user.username}!')
except AuthenticationError as e:
    print(f'Login failed: {e}')

MATHS_COMPETITION_ID = 3
LABELS = ['A', 'B', 'C', 'D']
QUESTIONS_PER_GAME = 15

In [ ]:
# Check GPU availability — required for fine-tuning, this is.
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU.')

props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
%%capture
# Install Unsloth and dependencies — takes ~2 minutes, it does.
import torch, re, os
v = re.match(r'[0-9\.]{3,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + ('0.0.32.post2' if v == '2.8.0' else '0.0.29.post3')
os.system(f'pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo -q')
os.system('pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer -q')
os.system('pip install --no-deps unsloth -q')
print('Unsloth installed.')

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Load Qwen3-4B with 4-bit quantization — fits T4 VRAM, this does.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen3-4B-Thinking-2507',
    max_seq_length = 2048,
    load_in_4bit   = True,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen-3')

# LoRA adapters attached — only a small fraction of parameters trained, they are.
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
print('Model loaded and LoRA adapters attached.')

In [ ]:
from datasets import load_dataset

def _format_math_conversation(examples):
    # Format problem+solution as chat turns, we do.
    convs = [
        [{'role': 'user',      'content': p},
         {'role': 'assistant', 'content': s}]
        for p, s in zip(examples['problem'], examples['generated_solution'])
    ]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convs
    ]
    return {'conversations': convs, 'text': texts}

math_ds = load_dataset('unsloth/OpenMathReasoning-mini', split='cot')
math_ds = math_ds.map(_format_math_conversation, batched=True)
print(f'Dataset ready: {len(math_ds)} examples.')
print('Sample:', math_ds[0]['problem'][:120])

In [ ]:
from trl import SFTTrainer, SFTConfig

ft_args = SFTConfig(
    dataset_text_field          = 'text',
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = 5,
    max_steps                   = 120,
    learning_rate               = 2e-4,
    logging_steps               = 10,
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'linear',
    seed                        = 42,
    output_dir                  = '/content/qwen3-math-finetuned',
    report_to                   = 'none',
)
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = math_ds,
    args          = ft_args,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part    = '<|im_start|>assistant\n',
)

mem_before = round(torch.cuda.max_memory_reserved() / 1e9, 2)
print(f'Training begins. GPU: {torch.cuda.get_device_name(0)} | Reserved: {mem_before} GB')

stats = trainer.train()

mem_after = round(torch.cuda.max_memory_reserved() / 1e9, 2)
print(f'Done in {stats.metrics["train_runtime"]:.0f}s '
      f'({stats.metrics["train_runtime"]/60:.1f} min). '
      f'Peak VRAM: {mem_after} GB | Loss: {stats.training_loss:.4f}')

In [ ]:
# Save fine-tuned weights to Drive — lost on runtime reset, they would be otherwise.
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')

In [ ]:
import random, re, time
from datetime import datetime

# Switch to inference mode — faster generation, this enables.
FastLanguageModel.for_inference(model)


def _parse_letter(raw: str, question):
    upper = raw.strip().upper()
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', upper)
    if m:
        chosen = m.group(1)
    else:
        m = re.search(r'\b([A-D])\b\s*$', upper)
        chosen = m.group(1) if m else None
    if chosen is None:
        for c in reversed(upper):
            if c in LABELS:
                chosen = c
                break
    if chosen is None:
        return random.choice(question.options).id
    return question.options[LABELS.index(chosen)].id


def finetuned_math_strategy(question):
    # Fine-tuned model answers math questions, it does. Wiser than before, it is.
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    messages = [{'role': 'user', 'content': (
        'Answer this multiple-choice math question.\n'
        'Think step by step, then end with: Answer: X  (X = A, B, C, or D)\n\n'
        f'Question: {question.text}\n\n{opts}'
    )}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=400,
            temperature=0.6, top_p=0.95, top_k=20, do_sample=True,
        )
    raw = tokenizer.decode(output[0], skip_special_tokens=True)
    return _parse_letter(raw, question)


def play_game(strategy_fn, game_index, verbose=True):
    game    = client.game.start(MATHS_COMPETITION_ID)
    correct = 0
    times   = []
    while game.in_progress:
        q = game.current_question
        if not q:
            break
        t0         = time.time()
        chosen_id  = strategy_fn(q)
        elapsed    = time.time() - t0
        result     = game.answer(chosen_id)
        times.append(elapsed)
        if result.correct:
            correct += 1
        if result.game_over:
            break
    accuracy = correct / QUESTIONS_PER_GAME
    if verbose:
        icon = '✅' if game.earned_amount > 0 else '❌'
        print(f'  {icon} Game {game_index} | Level {game.current_level} | '
              f'Earned: {game.earned_amount:,.0f} | Acc: {accuracy:.0%}')
    return {
        'game':              game_index,
        'timestamp':         datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        'correct':           correct,
        'accuracy':          round(accuracy, 4),
        'earned':            game.earned_amount,
        'reached_level':     game.current_level,
        'response_time_avg': round(sum(times) / len(times), 3) if times else 0.0,
    }


print('Inference ready. Run the next cell to evaluate.')

In [ ]:
import json

N_GAMES    = 5
STRAT_NAME = 'finetuned-qwen3-math'
LOG_FILE   = f'{NLP_DIR}/runs_log_colab.json'

print(f'=== Evaluating {STRAT_NAME} on Maths ({N_GAMES} games) ===')
games = []
for i in range(1, N_GAMES + 1):
    g = play_game(finetuned_math_strategy, game_index=i)
    games.append(g)

avg_acc    = sum(g['accuracy'] for g in games) / len(games)
avg_earned = sum(g['earned']   for g in games) / len(games)
avg_level  = sum(g['reached_level'] for g in games) / len(games)
print(f'\nResults: acc={avg_acc:.0%} | avg_earned={avg_earned:,.0f} | avg_level={avg_level:.1f}')

# Save results to Drive so they can be merged back into the main log.
log = json.loads(open(LOG_FILE).read()) if os.path.exists(LOG_FILE) else {}
log.setdefault(STRAT_NAME, {}).setdefault('Maths', {'runs': []})
run_id = len(log[STRAT_NAME]['Maths']['runs']) + 1
log[STRAT_NAME]['Maths']['runs'].append({
    'run_id':       run_id,
    'games':        games,
    'avg_accuracy': round(avg_acc, 4),
    'avg_earned':   round(avg_earned, 2),
})
with open(LOG_FILE, 'w') as f:
    json.dump(log, f, indent=2)
print(f'Results saved to {LOG_FILE}')